In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import OneHotEncoder

from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


In [59]:
# Define the dataset path
file_path = Path("../dataset/raw/bank-full.csv")

# Load the dataset
df = pd.read_csv(file_path, sep=";")

print("Dataset loaded successfully!")

Dataset loaded successfully!


## Remove duration Attribute

In [60]:
# Create a copy of the original dataset
df_processed = df.copy()

# Drop the duration column
df_processed = df_processed.drop(columns=["duration"])

In [61]:
# Display the updated dataset shape
print("Original dataset shape:", df.shape)
print("Processed dataset shape:", df_processed.shape)

# Check whether duration still exists
print("\nDoes duration exist?")
print("duration" in df_processed.columns)

# Display remaining columns
print("\nRemaining columns:")
print(df_processed.columns.tolist())

Original dataset shape: (45211, 17)
Processed dataset shape: (45211, 16)

Does duration exist?
False

Remaining columns:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome', 'y']


## Create previously_contacted Attribute

In [62]:
# Examine the pdays variable
print("pdays value distribution:")
print(df_processed["pdays"].value_counts().sort_index().head(20))

# Count customers who have never been contacted
never_contacted = (df_processed["pdays"] == -1).sum()

# Count customers who have previously been contacted
previously_contacted = (df_processed["pdays"] != -1).sum()

print("\nContact history:")
print("Never previously contacted:", never_contacted)
print("Previously contacted:", previously_contacted)

pdays value distribution:
pdays
-1     36954
 1        15
 2        37
 3         1
 4         2
 5        11
 6        10
 7         7
 8        25
 9        12
 10        6
 12        7
 13        6
 14        9
 15        3
 17        4
 18        1
 19        2
 20        2
 21        4
Name: count, dtype: int64

Contact history:
Never previously contacted: 36954
Previously contacted: 8257


In [63]:
# Create a binary feature to indicate previous contact history
df_processed["previously_contacted"] = (
    df_processed["pdays"] != -1
).astype(int)

# Verify the newly created feature
print(df_processed["previously_contacted"].value_counts())

previously_contacted
0    36954
1     8257
Name: count, dtype: int64


# Replace -1 in pdays Attribute with NaN

In [64]:
# Replace -1 with NaN because it does not represent actual elapsed days
df_processed["pdays"] = df_processed["pdays"].replace(-1, np.nan)

# Check the updated values
print("Missing values in pdays:", df_processed["pdays"].isnull().sum())

Missing values in pdays: 36954


In [65]:
# Verify the updated dataset
print("Dataset shape:", df_processed.shape)

print("\nMissing values:")
print(df_processed[["pdays", "previously_contacted"]].isnull().sum())

print("\nPreviously contacted distribution:")
print(df_processed["previously_contacted"].value_counts())

print("\nSample records:")
print(df_processed[["pdays", "previously_contacted"]].head(10))

Dataset shape: (45211, 17)

Missing values:
pdays                   36954
previously_contacted        0
dtype: int64

Previously contacted distribution:
previously_contacted
0    36954
1     8257
Name: count, dtype: int64

Sample records:
   pdays  previously_contacted
0    NaN                     0
1    NaN                     0
2    NaN                     0
3    NaN                     0
4    NaN                     0
5    NaN                     0
6    NaN                     0
7    NaN                     0
8    NaN                     0
9    NaN                     0


## Train-Test Split

In [66]:
from sklearn.model_selection import train_test_split

# Separate predictor and target variables
X = df_processed.drop(columns=["y"])
y = df_processed["y"].map({
    "no": 0,
    "yes": 1
})

# Verify target encoding
print("Encoded target distribution:")
print(y.value_counts().sort_index())

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

Encoded target distribution:
y
0    39922
1     5289
Name: count, dtype: int64


In [67]:
# Identify categorical columns
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

# Identify numerical columns
numerical_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("Numerical columns:")
print(numerical_cols)


print("\nNumber of categorical columns:", len(categorical_cols))
print("\nNumber of numerical columns:", len(numerical_cols))

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical columns:
['age', 'balance', 'day', 'campaign', 'pdays', 'previous', 'previously_contacted']

Number of categorical columns: 9

Number of numerical columns: 7


In [68]:
# Check target data type
print("Target data type:", y.dtype)

# Check unique target values
print("Unique target values:", sorted(y.unique()))

# Check for missing values
print("Missing target values:", y.isnull().sum())

# Display class distribution
print("\nTarget class distribution:")
print(y.value_counts())

# Display class percentages
print("\nTarget class percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Target data type: int64
Unique target values: [np.int64(0), np.int64(1)]
Missing target values: 0

Target class distribution:
y
0    39922
1     5289
Name: count, dtype: int64

Target class percentages:
y
0    88.3
1    11.7
Name: proportion, dtype: float64


In [69]:
print("Predictor dataset shape:", X.shape)
print("Target dataset shape:", y.shape)

print("\nCategorical features:", len(categorical_cols))
print("Numerical features:", len(numerical_cols))

print("\nTarget distribution:")
print(y.value_counts())

Predictor dataset shape: (45211, 16)
Target dataset shape: (45211,)

Categorical features: 9
Numerical features: 7

Target distribution:
y
0    39922
1     5289
Name: count, dtype: int64


# Save the train-test datasets

In [70]:
import os
os.makedirs("../dataset/processed", exist_ok=True)

# Save training and testing datasets

X_train.to_csv("../dataset/processed/X_train.csv", index=False)
X_test.to_csv("../dataset/processed/X_test.csv", index=False)

y_train.to_csv("../dataset/processed/y_train.csv", index=False)
y_test.to_csv("../dataset/processed/y_test.csv", index=False)